# 05 · Selección del modelo ganador, métrica de negocio y registro

1. Ganador y segundo mejor con la evidencia de la CV (media, desviación y comparación pareada por pliegue).
2. Evaluación **una sola vez** en el hold-out (fechas objetivo 2025-09-01 → 2026-08-31).
3. Umbral de decisión por costo (FN:FP = 5:1), elegido con predicciones fuera de pliegue de la CV — nunca con el test.
4. Costo y explicabilidad.
5. Registro en el Model Registry (`ecu911-modelo`, alias `champion` / `challenger`), reproducción de la métrica desde el registro y exportación de todas las corridas a `docs/corridas.csv`.

In [ ]:
import sys, warnings
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")
FIG = RAIZ / "docs" / "figuras"
FIG.mkdir(parents=True, exist_ok=True)

def guardar(fig, nombre):
    fig.savefig(FIG / f"{nombre}.png", dpi=130, bbox_inches="tight")

In [ ]:
import time, pickle
from mlflow import MlflowClient
from sklearn.base import clone
from sklearn.inspection import permutation_importance

from src.datos import cargar_obt, construir_dataset, dividir_train_test, umbrales_p90, etiquetar
from src.pipeline import (ENTRADA, TARGET, SEMILLA, EXPERIMENTO, NOMBRE_MODELO, evaluar_cv, pr_auc,
                          reduccion_fn_vs_regla, registrar_en_corrida_activa, configurar_mlflow, contexto_git)

COSTO_FN, COSTO_FP = 5, 1          # decisión del equipo: no anticipar un pico cuesta 5 revisiones innecesarias
obt, meta = cargar_obt()
train, test = dividir_train_test(construir_dataset(obt))
mlflow = configurar_mlflow()
client = MlflowClient()
exp = mlflow.get_experiment_by_name(EXPERIMENTO)

## 1. Ganador y segundo mejor según la CV
Se toma la mejor corrida de cada modelo (por defecto o ajustada) y se ordenan por PR-AUC medio en CV.

In [ ]:
runs = mlflow.search_runs([exp.experiment_id], filter_string="attributes.status = 'FINISHED'")
elegibles = runs[(runs["tags.fase"] == "candidato") |
                 ((runs["tags.fase"] == "tuning") & (runs["tags.rol"] == "padre_mejor_configuracion"))].copy()
elegibles["modelo_base"] = np.where(elegibles["tags.fase"] == "tuning", elegibles["tags.tuning_de"], elegibles["tags.modelo"])
elegibles["version"] = np.where(elegibles["tags.fase"] == "tuning", "ajustado", "por defecto")
mejor_por_modelo = (elegibles.sort_values("metrics.cv_pr_auc_mean", ascending=False)
                    .drop_duplicates("modelo_base"))
ranking = mejor_por_modelo[["modelo_base", "version", "tags.familia", "run_id", "metrics.cv_pr_auc_mean",
                            "metrics.cv_pr_auc_std", "metrics.train_pr_auc", "metrics.cv_reduccion_fn_mean"]]
ranking.columns = ["modelo", "version", "familia", "run_id", "cv_pr_auc", "cv_std", "train_pr_auc", "cv_reduccion_fn"]
ranking = ranking.reset_index(drop=True)
display(ranking)
ganador, segundo = ranking.iloc[0], ranking.iloc[1]
print(f"Ganador: {ganador.modelo} ({ganador.version}) | Segundo: {segundo.modelo} ({segundo.version})")

In [ ]:
def por_pliegue(run_id):
    h = client.get_metric_history(run_id, "pliegue_pr_auc_val")
    return pd.Series({m.step: m.value for m in h}).sort_index()

p1, p2 = por_pliegue(ganador.run_id), por_pliegue(segundo.run_id)
dif = p1 - p2
comp = pd.DataFrame({ganador.modelo: p1, segundo.modelo: p2, "diferencia": dif}).rename_axis("pliegue")
display(comp)
print(f"Diferencia de medias en CV: {ganador.cv_pr_auc - segundo.cv_pr_auc:+.4f} | "
      f"desviación entre pliegues: ganador {ganador.cv_std:.4f}, segundo {segundo.cv_std:.4f}")
print(f"Comparación pareada: diferencia media {dif.mean():+.4f} ± {dif.std(ddof=1):.4f}; "
      f"el ganador es mejor en {int((dif > 0).sum())} de {len(dif)} pliegues")
supera_std = (ganador.cv_pr_auc - segundo.cv_pr_auc) > max(ganador.cv_std, segundo.cv_std)
supera_pareado = (dif.mean() > dif.std(ddof=1)) and (dif > 0).sum() >= 4
print(f"¿La diferencia supera la variación entre pliegues? {supera_std} | ¿es consistente pliegue a pliegue? {supera_pareado}")

## 2. Umbral de decisión por costo (con predicciones fuera de pliegue)
Para cada modelo se recalculan las predicciones de validación de los 5 pliegues (mismo pipeline, misma CV) y se elige el umbral que **minimiza 5·FN + 1·FP**. El umbral se fija aquí y luego se aplica tal cual al test. Como referencia se muestran el umbral inicial del PSet 1 (0,70) y el umbral teórico de un modelo calibrado, 1 / (1 + 5) ≈ 0,167.

In [ ]:
def cargar(run_id):
    return mlflow.sklearn.load_model(f"runs:/{run_id}/modelo")

modelos = {"ganador": cargar(ganador.run_id), "segundo": cargar(segundo.run_id)}

def costo(y, alerta):
    y, alerta = np.asarray(y, bool), np.asarray(alerta, bool)
    fn, fp = int((y & ~alerta).sum()), int((~y & alerta).sum())
    return COSTO_FN * fn + COSTO_FP * fp, fn, fp

umbrales_decision, oof = {}, {}
grid = np.unique(np.concatenate([np.linspace(0.01, 0.99, 197), [0.70, 1 / 6]]))
for rol, m in modelos.items():
    r = evaluar_cv(clone(m), train, guardar_predicciones=True)
    pred = r.predicciones
    oof[rol] = pred
    costos = np.array([costo(pred[TARGET], pred["score"] >= u)[0] for u in grid])
    umbrales_decision[rol] = float(grid[costos.argmin()])
    print(f"{rol}: umbral que minimiza el costo en CV = {umbrales_decision[rol]:.3f} "
          f"(costo {costos.min():,} vs {costo(pred[TARGET], pred['regla'])[0]:,} de la regla en los mismos pliegues)")

fig, ax = plt.subplots(figsize=(9, 4))
for rol in modelos:
    pred = oof[rol]
    ax.plot(grid, [costo(pred[TARGET], pred["score"] >= u)[0] for u in grid], label=f"{rol}")
    ax.axvline(umbrales_decision[rol], ls=":", color=ax.lines[-1].get_color())
ax.axhline(costo(oof["ganador"][TARGET], oof["ganador"]["regla"])[0], color="k", ls="--", label="regla estacional")
ax.axvline(0.70, color="gray", ls="-.", label="umbral inicial PSet 1 (0,70)")
ax.set(xlabel="umbral de probabilidad", ylabel="costo = 5·FN + FP (validación, 5 pliegues)",
       title="Costo del error según el umbral"); ax.legend(fontsize=8)
fig.tight_layout(); guardar(fig, "05_costo_vs_umbral")

## 3. Evaluación única en el conjunto de prueba
El target del test usa el umbral P90 aprendido con **todo el entrenamiento** (el mismo que vio el modelo final). Se evalúan solo el ganador y el segundo mejor; la regla estacional se calcula como referencia de negocio (no se elige nada con ella).

In [ ]:
te = etiquetar(test, umbrales_p90(train))
regla_test = (te["media_mismo_dia_4s"] > te["umbral_p90"]).to_numpy()
resultados_test, scores_test = [], {}
for rol, m in modelos.items():
    t0 = time.perf_counter()
    s = m.predict_proba(te[ENTRADA])[:, 1]
    seg_inf = time.perf_counter() - t0
    scores_test[rol] = s
    u = umbrales_decision[rol]
    c_mod, fn, fp = costo(te[TARGET], s >= u)
    c_reg, fn_r, fp_r = costo(te[TARGET], regla_test)
    neg = reduccion_fn_vs_regla(te[TARGET], s, regla_test)
    alerta = s >= u
    resultados_test.append({
        "rol": rol, "modelo": (ganador if rol == "ganador" else segundo).modelo,
        "test_pr_auc": pr_auc(te[TARGET], s), "umbral": u,
        "recall": float((alerta & te[TARGET].astype(bool)).sum() / te[TARGET].sum()),
        "precision": float((alerta & te[TARGET].astype(bool)).sum() / max(alerta.sum(), 1)),
        "alertas_por_dia": alerta.sum() / te["fecha"].nunique(),
        "FN": fn, "FP": fp, "costo": c_mod, "costo_regla": c_reg,
        "ahorro_costo_vs_regla": 1 - c_mod / c_reg,
        "reduccion_fn_mismas_alertas": neg["reduccion_fn"],
        "seg_inferencia_test": seg_inf, "ms_por_1000_filas": 1000 * seg_inf / len(te) * 1000,
    })
res_test = pd.DataFrame(resultados_test)
print(f"Test: {len(te):,} filas, prevalencia {te[TARGET].mean():.4f}; regla estacional: PR-AUC "
      f"{pr_auc(te[TARGET], te['media_mismo_dia_4s'] / (te['media_mismo_dia_4s'] + te['umbral_p90'])):.4f}, "
      f"recall {(regla_test & te[TARGET].astype(bool)).sum() / te[TARGET].sum():.3f}, "
      f"{regla_test.sum() / te['fecha'].nunique():.1f} alertas/día")
res_test

In [ ]:
# Incertidumbre del test: bootstrap por días (se remuestrean fechas completas, 500 repeticiones)
rng = np.random.default_rng(SEMILLA)
fechas = te["fecha"].to_numpy()
dias = np.unique(fechas)
idx_por_dia = {d: np.flatnonzero(fechas == d) for d in dias}
y = te[TARGET].to_numpy()
boot = []
for _ in range(500):
    idx = np.concatenate([idx_por_dia[d] for d in rng.choice(dias, len(dias), replace=True)])
    boot.append([pr_auc(y[idx], scores_test["ganador"][idx]), pr_auc(y[idx], scores_test["segundo"][idx])])
boot = np.array(boot)
ic = pd.DataFrame({"PR-AUC test": [pr_auc(y, scores_test["ganador"]), pr_auc(y, scores_test["segundo"])],
                   "IC95% inferior": np.percentile(boot, 2.5, axis=0), "IC95% superior": np.percentile(boot, 97.5, axis=0)},
                  index=[ganador.modelo, segundo.modelo])
d = boot[:, 0] - boot[:, 1]
print(f"Diferencia ganador − segundo en test: {d.mean():+.4f} (IC95% {np.percentile(d, 2.5):+.4f} a {np.percentile(d, 97.5):+.4f})")
ic

In [ ]:
from sklearn.metrics import precision_recall_curve
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for rol in modelos:
    pr, rc, _ = precision_recall_curve(y, scores_test[rol])
    ax[0].plot(rc, pr, label=f"{rol}: {res_test.set_index('rol').loc[rol, 'modelo']}")
rr = (regla_test & y.astype(bool)).sum() / y.sum(); rp = (regla_test & y.astype(bool)).sum() / regla_test.sum()
ax[0].scatter([rr], [rp], color="k", zorder=5, label="regla estacional")
ax[0].axhline(y.mean(), color="gray", ls=":", label="azar")
ax[0].set(xlabel="recall", ylabel="precisión", title="Curva Precision-Recall en test"); ax[0].legend(fontsize=8)
tab = res_test.set_index("modelo")[["costo"]].assign(regla=res_test["costo_regla"].iloc[0])
pd.concat([tab["costo"], pd.Series({"regla estacional": tab["regla"].iloc[0]})]).plot.bar(ax=ax[1], color=["#4c72b0", "#55a868", "#999999"])
ax[1].set(ylabel="5·FN + FP en test", title="Costo de negocio en test (umbral elegido en CV)"); ax[1].tick_params(axis="x", rotation=0)
fig.tight_layout(); guardar(fig, "05_test_pr_y_costo")

## 4. Costo y explicabilidad

In [ ]:
filas = []
for rol, fila in [("ganador", ganador), ("segundo", segundo)]:
    mets = mlflow.get_run(fila.run_id).data.metrics
    filas.append({"modelo": fila.modelo, "seg_entrenamiento_final": mets.get("seg_entrenamiento_final"),
                  "seg_ajuste_por_pliegue": mets.get("seg_fit_promedio"),
                  "ms_por_1000_filas_inferencia": res_test.set_index("rol").loc[rol, "ms_por_1000_filas"],
                  "tamano_modelo_MB": len(pickle.dumps(modelos[rol])) / 1e6})
pd.DataFrame(filas)

In [ ]:
# ¿Por qué predice lo que predice? Importancia por permutación (caída de PR-AUC al desordenar cada columna
# de entrada) sobre una muestra del test, solo para explicar: la selección ya está hecha.
muestra = te.sample(30_000, random_state=SEMILLA)
pi = permutation_importance(modelos["ganador"], muestra[ENTRADA], muestra[TARGET],
                            scoring="average_precision", n_repeats=5, random_state=SEMILLA, n_jobs=1)
imp = pd.Series(pi.importances_mean, index=ENTRADA).sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
imp.plot.barh(ax=ax, xerr=pd.Series(pi.importances_std, index=ENTRADA)[imp.index], color="#4c72b0")
ax.set(xlabel="caída de PR-AUC al permutar la columna", title=f"Importancia por permutación — {ganador.modelo}")
fig.tight_layout(); guardar(fig, "05_importancia_permutacion")
imp.sort_values(ascending=False).to_frame("caida_pr_auc").head(10)

In [ ]:
# Ejemplo local: la alerta de mayor probabilidad del test y sus valores relativos a su umbral
fila = te.assign(score=scores_test["ganador"]).sort_values("score", ascending=False).iloc[0]
X1 = modelos["ganador"].named_steps["perfil"].transform(te.loc[[fila.name], ENTRADA])
print(f"{fila['cod_canton']} {obt.loc[obt.cod_canton == fila['cod_canton'], 'canton'].iloc[0]} | t = {fila['fecha'].date()} "
      f"-> t+3 = {fila['fecha_obj'].date()} | prob = {fila['score']:.3f} | real en t+3 = {int(fila['obj_n_emergencias_t3'])} "
      f"vs P90 = {fila['umbral_p90']:.1f} (y = {int(fila[TARGET])})")
X1[[c_ for c_ in X1.columns if c_.startswith("rel_")] + ["p90_obj", "es_feriado_t3", "dia_semana_obj"]].T

## 5. Registro en el Model Registry
La corrida **final** (`fase=final`) reentrena el ganador con todo el entrenamiento (es determinista: misma semilla que la corrida de tuning), guarda el pipeline y registra las métricas de test, el umbral de decisión y la métrica de negocio. El segundo mejor se registra desde su corrida. Si este notebook se vuelve a ejecutar con el mismo commit, se reutilizan la corrida final y las versiones ya registradas.

In [ ]:
git = contexto_git()
rt = res_test.set_index("rol")
previas = mlflow.search_runs([exp.experiment_id], filter_string=(
    f"tags.fase = 'final' and tags.modelo = '{ganador.modelo}' and tags.git_commit = '{git['git_commit']}'"
    " and attributes.status = 'FINISHED'"))
if len(previas):
    run_final = previas.iloc[0]["run_id"]
    print("Se reutiliza la corrida final existente:", run_final)
else:
    r_cv = evaluar_cv(clone(modelos["ganador"]), train)
    with mlflow.start_run(run_name=f"final_{ganador.modelo}") as run:
        final = registrar_en_corrida_activa(
            clone(modelos["ganador"]), train, meta, fase="final", familia=ganador.familia, resultado=r_cv,
            params={"run_id_origen": ganador.run_id, "umbral_decision": rt.loc["ganador", "umbral"],
                    "costo_fn": COSTO_FN, "costo_fp": COSTO_FP},
            tags_extra={"modelo": ganador.modelo, "rol": "champion"},
            metricas_extra={"test_recall": rt.loc["ganador", "recall"], "test_precision": rt.loc["ganador", "precision"],
                            "test_costo": rt.loc["ganador", "costo"], "test_costo_regla": rt.loc["ganador", "costo_regla"],
                            "test_ahorro_costo_vs_regla": rt.loc["ganador", "ahorro_costo_vs_regla"],
                            "test_reduccion_fn_mismas_alertas": rt.loc["ganador", "reduccion_fn_mismas_alertas"],
                            "test_alertas_por_dia": rt.loc["ganador", "alertas_por_dia"],
                            "test_prevalencia": float(te[TARGET].mean())})
        # La métrica registrada sale del pipeline guardado en ESTA corrida (el que se registra)
        pr_final = pr_auc(te[TARGET], final.predict_proba(te[ENTRADA])[:, 1])
        mlflow.log_metric("test_pr_auc", pr_final)
        print(f"PR-AUC test del modelo reentrenado en la corrida final: {pr_final:.10f} "
              f"(corrida de origen: {rt.loc['ganador', 'test_pr_auc']:.10f})")
    run_final = run.info.run_id
# El segundo mejor: métricas de test en su propia corrida
for k, v in {"test_pr_auc": rt.loc["segundo", "test_pr_auc"], "test_costo": rt.loc["segundo", "costo"],
             "test_ahorro_costo_vs_regla": rt.loc["segundo", "ahorro_costo_vs_regla"]}.items():
    client.log_metric(segundo.run_id, k, float(v))

def registrar_version(run_id, alias, extra):
    existentes = [v for v in client.search_model_versions(f"name = '{NOMBRE_MODELO}'") if v.run_id == run_id]
    v = existentes[0] if existentes else mlflow.register_model(f"runs:/{run_id}/modelo", NOMBRE_MODELO)
    client.set_registered_model_alias(NOMBRE_MODELO, alias, v.version)
    for k, val in extra.items():
        client.set_model_version_tag(NOMBRE_MODELO, v.version, k, str(val))
    return v.version

v_champion = registrar_version(run_final, "champion", {"modelo": ganador.modelo, "umbral_decision": rt.loc["ganador", "umbral"],
                                                        "test_pr_auc": round(rt.loc["ganador", "test_pr_auc"], 6)})
v_challenger = registrar_version(segundo.run_id, "challenger", {"modelo": segundo.modelo, "umbral_decision": rt.loc["segundo", "umbral"],
                                                                 "test_pr_auc": round(rt.loc["segundo", "test_pr_auc"], 6)})
print(f"{NOMBRE_MODELO}: champion = versión {v_champion} (run {run_final}) | challenger = versión {v_challenger} (run {segundo.run_id})")

## 6. Reproducción desde el registro (estado limpio)
Se ejecuta en un **proceso de Python nuevo** (sin ninguna variable de este notebook): carga `models:/ecu911-modelo@champion`, vuelve a leer la OBT, reconstruye el test y recalcula el PR-AUC; debe coincidir exactamente con `test_pr_auc` registrado en la corrida de la versión champion.

In [ ]:
import subprocess, sys, textwrap
script = textwrap.dedent(f"""
    import sys, warnings; warnings.filterwarnings("ignore"); sys.path.insert(0, r"{RAIZ}")
    import mlflow
    from src.datos import cargar_obt, construir_dataset, dividir_train_test, umbrales_p90, etiquetar
    from src.pipeline import ENTRADA, TARGET, NOMBRE_MODELO, pr_auc, configurar_mlflow
    configurar_mlflow()
    uri = f"models:/{{NOMBRE_MODELO}}@champion"
    modelo = mlflow.sklearn.load_model(uri)
    version = mlflow.MlflowClient().get_model_version_by_alias(NOMBRE_MODELO, "champion")
    registrada = mlflow.get_run(version.run_id).data.metrics["test_pr_auc"]
    obt, meta = cargar_obt()
    train, test = dividir_train_test(construir_dataset(obt))
    te = etiquetar(test, umbrales_p90(train))
    recalculada = pr_auc(te[TARGET], modelo.predict_proba(te[ENTRADA])[:, 1])
    print(f"{{uri}} -> versión {{version.version}} (run {{version.run_id}})")
    print(f"hash de los datos: {{meta['dataset_hash'][:16]}} | fuente: {{meta['fuente']}}")
    print(f"PR-AUC registrado = {{registrada:.10f}} | recalculado = {{recalculada:.10f}} | idénticos: {{abs(registrada - recalculada) < 1e-12}}")
""")
salida = subprocess.run([sys.executable, "-c", script], capture_output=True, text=True, encoding="utf-8",
                        env={**__import__("os").environ, "PYTHONIOENCODING": "utf-8"})
print(salida.stdout[-2000:] or salida.stderr[-3000:])

## 7. Exportar todas las corridas a `docs/corridas.csv`

In [ ]:
todas = mlflow.search_runs([exp.experiment_id], max_results=5000)
cols = (["run_id", "tags.mlflow.runName", "status", "start_time", "end_time", "tags.fase", "tags.familia",
         "tags.modelo", "tags.tuning_de", "tags.rol", "tags.mlflow.parentRunId", "tags.git_commit"]
        + sorted(c_ for c_ in todas.columns if c_.startswith("params."))
        + sorted(c_ for c_ in todas.columns if c_.startswith("metrics.")))
corridas = todas[[c_ for c_ in cols if c_ in todas.columns]].sort_values("start_time")
corridas.to_csv(RAIZ / "docs" / "corridas.csv", index=False)
print(f"docs/corridas.csv: {len(corridas)} corridas x {corridas.shape[1]} columnas")
corridas.groupby("tags.fase", dropna=False).size().rename("corridas").to_frame()